# 

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.1 MB/s eta 0:00:00


In [4]:
!pip install deep-sort-realtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 59.4 MB/s eta 0:00:0000:0100:01


In [5]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        PERSON DISAPPEARANCE DETECTION SYSTEM  ·  Kaggle Edition            ║
║        YOLOv8 + Deep SORT  |  Boundary · Occlusion · Sudden Exit           ║
╚══════════════════════════════════════════════════════════════════════════════╝

Pipeline:
  1. YOLOv8  → detects persons every frame
  2. DeepSORT → maintains consistent IDs across frames
  3. Monitor every track; on loss → check boundary / occlusion / sudden
  4. Ghost dot + fading box drawn at last known location
  5. Top-RIGHT panel lists all disappeared IDs (small, clean)
  6. All events logged → CSV + JSON under OUTPUT_DIR/logs/

Kaggle paths  (edit the three lines in ── KAGGLE PATHS ──):
    INPUT_VIDEO  – your source clip
    OUTPUT_VIDEO – annotated result
    OUTPUT_DIR   – folder for logs

Install (Kaggle cell):
    !pip install ultralytics deep-sort-realtime -q
"""

# ─────────────────────────────────────────────────────────────────────────────
# ── KAGGLE PATHS  ◄  EDIT THESE THREE LINES
# ─────────────────────────────────────────────────────────────────────────────
INPUT_VIDEO  = "/kaggle/input/datasets/sarvada2223/ddataa/Guyswalking_video.mp4"
OUTPUT_VIDEO = "/kaggle/working/output_tracked_NEW_GUYS.mp4"
OUTPUT_DIR   = "/kaggle/working/disappearance_logs"
# ─────────────────────────────────────────────────────────────────────────────

import csv, json, os, time
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np


# ══════════════════════════════════════════════════════════════════════════════
# ENUMS
# ══════════════════════════════════════════════════════════════════════════════

class TrackStatus(Enum):
    ACTIVE      = "ACTIVE"
    LOST        = "LOST"
    EXITED      = "EXITED"
    OCCLUDED    = "OCCLUDED"
    DISAPPEARED = "DISAPPEARED"

class DisappearReason(Enum):
    BOUNDARY_EXIT = "BOUNDARY_EXIT"
    OCCLUSION     = "OCCLUSION"
    SUDDEN        = "SUDDEN_DISAPPEARANCE"

PERSON_CLASS = 0
FONT         = cv2.FONT_HERSHEY_SIMPLEX


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG  (tweak freely)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # YOLO
    yolo_model    : str   = "yolov8n.pt"
    conf          : float = 0.40
    iou_nms       : float = 0.45
    # SORT
    max_age       : int   = 40      # frames tracker lives without a detection
    n_init        : int   = 3       # frames to confirm a new track
    cosine_dist   : float = 0.40
    # Disappearance
    lost_thr      : int   = 10      # frames → LOST
    disappear_thr : int   = 30      # frames → classify & alert
    boundary_px   : int   = 45      # px from edge = boundary zone
    occ_iou       : float = 0.35    # IOU with active bbox → occluded
    # Visuals
    trail_len     : int   = 60
    ghost_ttl     : float = 7.0     # seconds ghost stays on screen

CFG = Config()


# ══════════════════════════════════════════════════════════════════════════════
# DATA MODELS
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class BBox:
    x1: int; y1: int; x2: int; y2: int

    @property
    def center(self) -> Tuple[int, int]:
        return ((self.x1 + self.x2) // 2, (self.y1 + self.y2) // 2)

    @property
    def area(self) -> float:
        return max(0, self.x2 - self.x1) * max(0, self.y2 - self.y1)

    def iou(self, other: "BBox") -> float:
        ix1 = max(self.x1, other.x1); iy1 = max(self.y1, other.y1)
        ix2 = min(self.x2, other.x2); iy2 = min(self.y2, other.y2)
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        union = self.area + other.area - inter
        return inter / union if union > 0 else 0.0

    def as_tuple(self):
        return (self.x1, self.y1, self.x2, self.y2)

    def to_dict(self):
        return {"x1": self.x1, "y1": self.y1, "x2": self.x2, "y2": self.y2,
                "center_x": self.center[0], "center_y": self.center[1]}


@dataclass
class DisappearEvent:
    person_id    : int
    frame_no     : int
    timestamp    : str
    reason       : str
    last_bbox    : dict
    last_center  : Tuple[int, int]
    video_sec    : float
    trail        : List[Tuple[int, int]]


@dataclass
class TrackRecord:
    track_id     : int
    status       : TrackStatus              = TrackStatus.ACTIVE
    last_bbox    : Optional[BBox]           = None
    last_seen    : int                      = 0
    first_seen   : int                      = 0
    missing      : int                      = 0
    trail        : deque                    = field(default_factory=lambda: deque(maxlen=CFG.trail_len))
    color        : Tuple[int, int, int]     = (0, 255, 0)
    event        : Optional[DisappearEvent] = None
    ghost_until  : float                    = 0.0       # wall-clock expiry


# ══════════════════════════════════════════════════════════════════════════════
# COLOUR UTILITY
# ══════════════════════════════════════════════════════════════════════════════

def track_color(tid: int) -> Tuple[int, int, int]:
    rng = np.random.default_rng(tid * 2654435761 % (2 ** 32))
    h   = int(rng.integers(0, 180))
    bgr = cv2.cvtColor(np.uint8([[[h, 215, 205]]]), cv2.COLOR_HSV2BGR)[0][0]
    return (int(bgr[0]), int(bgr[1]), int(bgr[2]))


# ══════════════════════════════════════════════════════════════════════════════
# LOGGER
# ══════════════════════════════════════════════════════════════════════════════

class EventLogger:
    def __init__(self, log_dir: str):
        p = Path(log_dir); p.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.csv_path  = p / f"events_{ts}.csv"
        self.json_path = p / f"events_{ts}.json"
        self._events: List[dict] = []
        with open(self.csv_path, "w", newline="") as f:
            csv.writer(f).writerow([
                "person_id","frame","timestamp","video_sec",
                "reason","x1","y1","x2","y2","cx","cy"
            ])
        print(f"  CSV  → {self.csv_path}")
        print(f"  JSON → {self.json_path}")

    def log(self, ev: DisappearEvent):
        b = ev.last_bbox
        row = dict(person_id=ev.person_id, frame=ev.frame_no,
                   timestamp=ev.timestamp, video_sec=ev.video_sec,
                   reason=ev.reason,
                   x1=b["x1"], y1=b["y1"], x2=b["x2"], y2=b["y2"],
                   cx=b["center_x"], cy=b["center_y"])
        self._events.append(row)
        with open(self.csv_path, "a", newline="") as f:
            csv.writer(f).writerow(row.values())
        with open(self.json_path, "w") as f:
            json.dump(self._events, f, indent=2)
        print(f"\n  ⚠  DISAPPEARANCE  ID:{ev.person_id}  "
              f"frame:{ev.frame_no}  reason:{ev.reason}  "
              f"center:{ev.last_center}")


# ══════════════════════════════════════════════════════════════════════════════
# SCENE ANALYSER
# ══════════════════════════════════════════════════════════════════════════════

class SceneAnalyzer:
    def __init__(self, w: int, h: int, margin: int):
        self.w, self.h, self.m = w, h, margin

    def near_boundary(self, b: BBox) -> bool:
        m = self.m
        return (b.x1 < m or b.y1 < m or
                b.x2 > self.w - m or b.y2 > self.h - m)

    def occluded(self, b: BBox, others: List[BBox]) -> bool:
        return any(b.iou(o) >= CFG.occ_iou for o in others)

    def classify(self, b: BBox, others: List[BBox]) -> DisappearReason:
        if self.near_boundary(b):   return DisappearReason.BOUNDARY_EXIT
        if self.occluded(b, others): return DisappearReason.OCCLUSION
        return DisappearReason.SUDDEN


# ══════════════════════════════════════════════════════════════════════════════
# DRAWING HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def pill(img, text, x, y, bg, fg=(255, 255, 255), scale=0.50, thick=1):
    """Text with filled pill background."""
    (tw, th), bl = cv2.getTextSize(text, FONT, scale, thick)
    pad = 4
    cv2.rectangle(img, (x - pad, y - th - pad - bl),
                  (x + tw + pad, y + pad - bl), bg, -1)
    cv2.putText(img, text, (x, y - bl), FONT, scale, fg, thick, cv2.LINE_AA)


def draw_active(frame, rec: TrackRecord):
    if rec.last_bbox is None: return
    x1, y1, x2, y2 = rec.last_bbox.as_tuple()
    col = rec.color
    # box
    cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
    pill(frame, f" ID {rec.track_id} ", x1, y1 - 2, col)
    # centre dot
    cv2.circle(frame, rec.last_bbox.center, 4, col, -1)


def draw_ghost(frame, rec: TrackRecord, now: float) -> bool:
    """
    Draw fading ghost at last known location.
    Returns True while still alive, False when TTL expired.
    """
    if rec.last_bbox is None or now > rec.ghost_until:
        return False

    remaining = rec.ghost_until - now
    fade      = min(1.0, remaining / CFG.ghost_ttl)   # 1 → 0

    x1, y1, x2, y2 = rec.last_bbox.as_tuple()
    cx, cy = rec.last_bbox.center
    reason  = rec.event.reason if rec.event else "?"

    # colour by reason
    if reason == DisappearReason.SUDDEN.value:
        gc = (0, 0, 230)          # red-ish
    elif reason == DisappearReason.OCCLUSION.value:
        gc = (0, 140, 255)        # orange
    else:
        gc = (160, 160, 0)        # teal (boundary)

    ov = frame.copy()

    # faded bounding box
    cv2.rectangle(ov, (x1, y1), (x2, y2), gc, 2)

    # pulsing ring + solid dot  ← THE DOT for last stored location
    r     = max(10, (x2 - x1) // 4)
    pulse = int(r * (0.82 + 0.18 * np.sin(now * 7)))
    cv2.circle(ov, (cx, cy), pulse, gc, 2)
    cv2.circle(ov, (cx, cy), 6, gc, -1)          # ← solid dot

    # crosshair
    cv2.line(ov, (cx - r - 6, cy), (cx + r + 6, cy), gc, 1)
    cv2.line(ov, (cx, cy - r - 6), (cx, cy + r + 6), gc, 1)

    # label beneath ghost box
    pill(ov, f" ID {rec.track_id}  {reason} ", x1, y2 + 18, gc, scale=0.46)
    pill(ov, f" LAST ({cx},{cy}) ",            x1, y2 + 36,
         (30, 30, 30), fg=(200, 200, 200), scale=0.42)

    cv2.addWeighted(ov, fade * 0.85, frame, 1 - fade * 0.85, 0, frame)
    return True


# ── TOP-RIGHT  disappeared-ID panel ──────────────────────────────────────────

def draw_disappeared_panel(frame, records: Dict[int, "TrackRecord"],
                           frame_w: int):
    """
    Small, clean panel at TOP-RIGHT listing disappeared IDs.
    Shows only IDs whose ghost is still alive OR were recently classified.
    """
    now = time.time()

    # collect recently disappeared (ghost still alive or within last 120s)
    entries = []
    for tid, rec in records.items():
        if rec.status in (TrackStatus.DISAPPEARED,
                          TrackStatus.EXITED,
                          TrackStatus.OCCLUDED):
            reason_short = {
                DisappearReason.SUDDEN.value:        "VANISHED",
                DisappearReason.BOUNDARY_EXIT.value: "EXITED",
                DisappearReason.OCCLUSION.value:     "OCCLUDED",
            }.get(rec.event.reason if rec.event else "", "?")
            entries.append((tid, reason_short, rec.color))

    if not entries:
        return

    # layout
    pad    = 6
    scale  = 0.42
    lh     = 17
    title  = "DISAPPEARED"
    (ttw, tth), _ = cv2.getTextSize(title, FONT, 0.45, 1)
    max_w  = ttw
    for tid, rs, _ in entries:
        (ew, _), _ = cv2.getTextSize(f"ID {tid}  {rs}", FONT, scale, 1)
        max_w = max(max_w, ew)

    panel_w = max_w + pad * 4
    panel_h = lh + lh * len(entries) + pad * 2 + 4

    x0 = frame_w - panel_w - 8
    y0 = 8

    # semi-transparent dark background
    ov = frame.copy()
    cv2.rectangle(ov, (x0, y0), (x0 + panel_w, y0 + panel_h), (18, 18, 18), -1)
    cv2.addWeighted(ov, 0.72, frame, 0.28, 0, frame)

    # thin accent border
    cv2.rectangle(frame, (x0, y0), (x0 + panel_w, y0 + panel_h), (0, 180, 255), 1)

    # title row
    cv2.putText(frame, title,
                (x0 + pad, y0 + pad + tth),
                FONT, 0.45, (0, 200, 255), 1, cv2.LINE_AA)

    # one row per ID
    for i, (tid, rs, col) in enumerate(entries):
        ty = y0 + pad + lh * (i + 2) + 2
        # small colour swatch
        cv2.rectangle(frame,
                      (x0 + pad, ty - lh + 5),
                      (x0 + pad + 6, ty - 3), col, -1)
        cv2.putText(frame, f"ID {tid}  {rs}",
                    (x0 + pad + 10, ty),
                    FONT, scale, col, 1, cv2.LINE_AA)


# ── Bottom status bar ─────────────────────────────────────────────────────────

def draw_status_bar(frame, frame_idx, fps, active, events, frame_h):
    bar_h = 24
    y0    = frame_h - bar_h
    ov    = frame.copy()
    cv2.rectangle(ov, (0, y0), (frame.shape[1], frame_h), (15, 15, 15), -1)
    cv2.addWeighted(ov, 0.70, frame, 0.30, 0, frame)
    txt = (f"  Frame {frame_idx}   |   FPS {fps:.1f}   |"
           f"   Active {active}   |   Events {events}")
    cv2.putText(frame, txt, (4, y0 + 16), FONT, 0.40,
                (200, 200, 200), 1, cv2.LINE_AA)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN SYSTEM
# ══════════════════════════════════════════════════════════════════════════════

class DisappearanceSystem:
    def __init__(self):
        self._load_yolo()
        self._load_deepsort()
        self.tracks  : Dict[int, TrackRecord] = {}
        self.ghosts  : List[TrackRecord]      = []
        self.logger  : Optional[EventLogger]  = None
        self.analyzer: Optional[SceneAnalyzer]= None
        self.frame_idx   = 0
        self.fps_display = 30.0
        self.total_events= 0

    # ── Init ──────────────────────────────────────────────────────────────────

    def _load_yolo(self):
        from ultralytics import YOLO
        self.yolo = YOLO(CFG.yolo_model)
        print(f"[✓] YOLOv8  loaded : {CFG.yolo_model}")

    def _load_deepsort(self):
        from deep_sort_realtime.deepsort_tracker import DeepSort
        self.tracker = DeepSort(
            max_age           = CFG.max_age,
            n_init            = CFG.n_init,
            max_cosine_distance = CFG.cosine_dist,
            nn_budget         = 100,
            embedder          = "mobilenet",
            half              = True,
            bgr               = True,
        )
        print("[✓] DeepSORT loaded")

    # ── Detection ─────────────────────────────────────────────────────────────

    def _detect(self, frame):
        res  = self.yolo(frame, conf=CFG.conf, iou=CFG.iou_nms,
                         classes=[PERSON_CLASS], verbose=False)
        dets = []
        if res and res[0].boxes is not None:
            for box in res[0].boxes:
                x1,y1,x2,y2 = map(int, box.xyxy[0].cpu().numpy())
                conf         = float(box.conf[0])
                dets.append(([x1, y1, x2-x1, y2-y1], conf, PERSON_CLASS))
        return dets

    # ── SORT update ───────────────────────────────────────────────────────────

    def _update(self, frame, raw) -> Dict[int, BBox]:
        active = {}
        for t in self.tracker.update_tracks(raw, frame=frame):
            if not t.is_confirmed(): continue
            l, tp, r, b = map(int, t.to_ltrb())
            active[int(t.track_id)] = BBox(l, tp, r, b)
        return active

    # ── Alert ─────────────────────────────────────────────────────────────────

    def _alert(self, rec: TrackRecord, others: List[BBox], vsec: float):
        reason = self.analyzer.classify(rec.last_bbox, others)
        ev     = DisappearEvent(
            person_id   = rec.track_id,
            frame_no    = self.frame_idx,
            timestamp   = datetime.now().isoformat(timespec="milliseconds"),
            reason      = reason.value,
            last_bbox   = rec.last_bbox.to_dict(),
            last_center = rec.last_bbox.center,
            video_sec   = round(vsec, 3),
            trail       = list(rec.trail),
        )
        rec.status     = {
            DisappearReason.BOUNDARY_EXIT: TrackStatus.EXITED,
            DisappearReason.OCCLUSION:     TrackStatus.OCCLUDED,
            DisappearReason.SUDDEN:        TrackStatus.DISAPPEARED,
        }[reason]
        rec.event      = ev
        rec.ghost_until= time.time() + CFG.ghost_ttl
        rec.trail.clear()          # ← wipe trail so no watery lines remain
        if self.logger: self.logger.log(ev)
        self.total_events += 1
        self.ghosts.append(rec)

    # ── Per-frame ─────────────────────────────────────────────────────────────

    def process(self, frame, vsec: float) -> np.ndarray:
        self.frame_idx += 1
        now = time.time()

        # 1. detect + track
        raw    = self._detect(frame)
        active = self._update(frame, raw)
        a_ids  = set(active.keys())
        a_bbs  = list(active.values())

        # 2. update records for visible tracks
        for tid, bbox in active.items():
            if tid not in self.tracks:
                self.tracks[tid] = TrackRecord(
                    track_id   = tid,
                    first_seen = self.frame_idx,
                    color      = track_color(tid),
                )
            r = self.tracks[tid]
            r.last_bbox  = bbox
            r.last_seen  = self.frame_idx
            r.missing    = 0
            r.trail.append(bbox.center)
            if r.status not in (TrackStatus.EXITED,
                                TrackStatus.OCCLUDED,
                                TrackStatus.DISAPPEARED):
                r.status = TrackStatus.ACTIVE

        # 3. handle missing tracks
        for tid, r in self.tracks.items():
            if tid in a_ids: continue
            if r.status in (TrackStatus.EXITED,
                            TrackStatus.OCCLUDED,
                            TrackStatus.DISAPPEARED):
                continue
            r.missing += 1
            if r.missing >= CFG.lost_thr:
                r.status = TrackStatus.LOST
            if r.missing == CFG.disappear_thr and r.last_bbox:
                self._alert(r, a_bbs, vsec)

        # 4. draw active tracks
        for tid in a_ids:
            if tid in self.tracks:
                draw_active(frame, self.tracks[tid])

        # 5. draw ghosts (prune expired)
        self.ghosts = [g for g in self.ghosts if draw_ghost(frame, g, now)]

        # 6. TOP-RIGHT disappeared panel
        draw_disappeared_panel(frame, self.tracks, frame.shape[1])

        # 7. bottom status bar
        draw_status_bar(frame, self.frame_idx, self.fps_display,
                        len(a_ids), self.total_events, frame.shape[0])

        return frame

    # ── Run ───────────────────────────────────────────────────────────────────

    def run(self, input_path: str, output_path: str, log_dir: str):
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open video: {input_path}")

        W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
        TOT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.fps_display = FPS
        self.analyzer    = SceneAnalyzer(W, H, CFG.boundary_px)
        self.logger      = EventLogger(log_dir)

        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        writer = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            FPS, (W, H)
        )

        print(f"\n[▶] Input   : {input_path}")
        print(f"[▶] Output  : {output_path}")
        print(f"[▶] Res     : {W}×{H}  |  FPS {FPS:.1f}  |  {TOT} frames\n")

        t0 = time.time(); fc = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            vsec  = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
            frame = self.process(frame, vsec)
            writer.write(frame)

            fc += 1
            elapsed = time.time() - t0
            if elapsed >= 1.0:
                self.fps_display = fc / elapsed
                fc, t0 = 0, time.time()

            if self.frame_idx % 100 == 0:
                pct = self.frame_idx / TOT * 100 if TOT else 0
                print(f"  [{self.frame_idx}/{TOT}  {pct:.1f}%]  "
                      f"fps={self.fps_display:.1f}  "
                      f"events={self.total_events}")

        cap.release(); writer.release()
        print(f"\n[✓] Done — {self.frame_idx} frames processed")
        print(f"[✓] Disappearance events : {self.total_events}")
        print(f"[✓] Output saved         : {output_path}")
        print(f"[✓] Logs saved           : {log_dir}/")


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    system = DisappearanceSystem()
    system.run(
        input_path  = INPUT_VIDEO,
        output_path = OUTPUT_VIDEO,
        log_dir     = OUTPUT_DIR,
    )

[✓] YOLOv8  loaded : yolov8n.pt
[✓] DeepSORT loaded


FileNotFoundError: Cannot open video: /kaggle/input/datasets/sarvada2223/ddataa/Guyswalking_video.mp4